# Task 2: Lightweight QLoRA Instruction Fine-Tuning

This notebook demonstrates a practical instruction fine-tuning workflow for a Senior Machine Learning Engineer assessment. The goal is not to chase benchmarks. The goal is to show reproducible, explainable, free-tier-compatible GenAI engineering.

We fine-tune a small instruct model with QLoRA, evaluate fixed prompts before and after tuning, save LoRA adapters locally, and optionally upload the adapter to Hugging Face Hub.

## 1. Introduction

QLoRA combines two practical ideas:

- **4-bit quantization**: load the base model with lower-precision weights to reduce GPU memory.
- **LoRA adapters**: train a small number of adapter parameters while keeping the base model frozen.

This makes instruction tuning possible on constrained GPUs such as free Colab runtimes.

## 2. Environment Setup

Use a GPU runtime for training. In Colab, install dependencies if needed. The helper module uses lazy imports, so it can be inspected on CPU, but model loading/training expects GPU-compatible `bitsandbytes`.

In [ ]:
# Colab only: uncomment if dependencies are missing.
# !pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece

from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "task2_genai" else Path.cwd()
TASK2_SRC = PROJECT_ROOT / "task2_genai" / "src"

if str(TASK2_SRC) not in sys.path:
    sys.path.insert(0, str(TASK2_SRC))

from task2_genai.src.finetuning_workflow import (
    compare_before_after,
    create_instruction_dataset,
    ensure_task2_directories,
    export_default_training_config,
    load_base_model_and_tokenizer,
    load_instruction_dataset_for_training,
    load_json,
    load_model_with_adapter,
    load_training_config,
    prepare_model_for_lora,
    push_adapter_to_hub,
    resolve_config_paths,
    save_evaluation_results,
    save_sample_prompts,
    train_qlora_adapters,
    generate_response,
)

ensure_task2_directories()
config = resolve_config_paths(export_default_training_config())
config

ModuleNotFoundError: No module named 'task2_genai'

## 3. Dataset Preparation

The dataset uses a simple instruction/response format. A small high-quality dataset is appropriate here because the goal is to teach response style and format, not broad new knowledge.

Each row has:

- `instruction`: user request
- `response`: desired assistant answer

During preprocessing, rows are converted to the model's chat template with `tokenizer.apply_chat_template()`.

In [ ]:
dataset_rows = create_instruction_dataset(Path(config["dataset_path"]))
sample_prompts = save_sample_prompts(Path(config["sample_prompts_path"]))

print(f"Training examples: {len(dataset_rows)}")
print(f"Evaluation prompts: {len(sample_prompts)}")
dataset_rows[:3]

## 4. Baseline Model Evaluation

Before training, generate baseline outputs on fixed prompts. Keeping prompts fixed makes the before/after comparison fair and reproducible.

In [ ]:
# This cell downloads the base model and should be run on a GPU runtime.
base_model, tokenizer = load_base_model_and_tokenizer(config)

In [ ]:
baseline_responses = []
for prompt in sample_prompts:
    response = generate_response(
        base_model,
        tokenizer,
        prompt,
        max_new_tokens=config["max_new_tokens"],
    )
    baseline_responses.append(response)
    print("PROMPT:", prompt)
    print("BASELINE:", response)
    print("-" * 80)

## 5. QLoRA Fine-Tuning

The model is prepared for k-bit training, then LoRA adapters are attached. Only adapter parameters are trained.

Important tradeoffs:

- Lower memory than full fine-tuning
- Smaller artifacts because adapters are saved separately
- Less flexible than updating all model weights
- Can overfit quickly on tiny datasets, so keep epochs conservative

In [ ]:
train_dataset = load_instruction_dataset_for_training(
    config["dataset_path"],
    tokenizer,
)
train_dataset[0]["text"][:500]

In [ ]:
qlora_model = prepare_model_for_lora(base_model, config)
qlora_model.print_trainable_parameters()

In [ ]:
trainer = train_qlora_adapters(
    qlora_model,
    tokenizer,
    train_dataset,
    config,
)

print("Saved adapter to:", config["adapter_output_dir"])

## 6. Post-Training Evaluation

Now generate responses from the adapter model using the same prompts. The comparison is qualitative because this dataset is intentionally small.

In [ ]:
# If you restarted the notebook after training, use:
# tuned_model, tokenizer = load_model_with_adapter(config)

tuned_model = qlora_model
fine_tuned_responses = []
for prompt in sample_prompts:
    response = generate_response(
        tuned_model,
        tokenizer,
        prompt,
        max_new_tokens=config["max_new_tokens"],
    )
    fine_tuned_responses.append(response)
    print("PROMPT:", prompt)
    print("FINE-TUNED:", response)
    print("-" * 80)

## 7. Sample Generations

The structured comparison below is saved under `task2_genai/evaluation/` for reproducibility.

In [ ]:
comparison_rows = compare_before_after(
    sample_prompts,
    baseline_responses,
    fine_tuned_responses,
)

evaluation_path = save_evaluation_results(
    comparison_rows,
    Path(config["evaluation_path"]),
)
print("Saved evaluation to:", evaluation_path)
comparison_rows

## 8. Saving Adapters

LoRA adapters are saved separately from the base model. This is practical because the adapter is much smaller than the full model and can be loaded on top of the original base model later.

In [ ]:
adapter_dir = Path(config["adapter_output_dir"])
print("Adapter directory:", adapter_dir)
print("Files:", list(adapter_dir.glob("*")) if adapter_dir.exists() else "Adapter not found yet")

## 9. Optional Hugging Face Upload

Upload only if you want to publish or back up the adapter. Authenticate first with `huggingface_hub.notebook_login()`.

In [ ]:
# Optional upload. Uncomment and set your repo id.
# from huggingface_hub import notebook_login
# notebook_login()
# push_adapter_to_hub(tuned_model, tokenizer, repo_id="your-username/task2-qwen-qlora-adapter", private=True)

## 10. Engineering Reflection

### Why QLoRA was chosen

QLoRA is suitable for free-tier GPUs because 4-bit quantization reduces memory usage while LoRA limits the number of trainable parameters.

### Why PEFT is practical

PEFT trains adapters instead of all model weights. This reduces GPU memory, training time, storage cost, and upload size.

### Free-tier GPU limitations

Free GPUs have limited VRAM, session timeouts, variable availability, and slower training. The config uses batch size 1, gradient accumulation, one epoch, and short sequence length to stay practical.

### Tradeoffs vs full fine-tuning

Full fine-tuning can adapt more deeply but requires much more compute and storage. LoRA is cheaper and easier to manage, but it may be less expressive for large behavior changes.

### Why small high-quality datasets can work

For style and format adaptation, clear examples often matter more than volume. A tiny dataset will not teach broad knowledge, but it can teach the expected response pattern.

### Common mistakes

- Training without baseline outputs
- Changing evaluation prompts after tuning
- Using too many epochs on a tiny dataset
- Forgetting to save the tokenizer with adapters
- Treating qualitative improvement as benchmark proof
- Running 4-bit training on CPU-only environments

### Hugging Face integration

The adapter can be pushed to the Hub after authentication. Users load the base model and then attach the adapter with PEFT.

### How to explain this in an interview

I built a reproducible QLoRA workflow: fixed config, small curated dataset, baseline generations, adapter training, post-training generations, and structured before/after evaluation. The design prioritizes practical fine-tuning on limited hardware and clear engineering artifacts over large-scale experimentation.